In [14]:
import requests
import re
import pandas as pd

In [15]:
#reading our api key file
with open('api.txt') as file:
    api_key = file.read().strip()
query = "\"Legumes and Legume Products\""

#creating a list from 5 pages of data from our API call
legus = []
for page in range(1,6):
    url = f'https://api.nal.usda.gov/fdc/v1/foods/list?api_key={api_key}&query={query}&pageNumber={page}'
    response = requests.get(url)
    legu_rows = response.json()
    legus.extend(legu_rows)

legusdf = pd.DataFrame(legus)

In [16]:
legusdf = legusdf[legusdf["dataType"] == "Foundation"]

legusdf = legusdf.reset_index(drop = True)

# legusdf.head(50)

In [17]:
def extract_nutrient(nutrient_list, target):
    if nutrient_list is None:
        return None
    if len(nutrient_list) == 0:
        return None
    for nutrient in nutrient_list:
        if nutrient.get('name') == target:
            return nutrient.get("amount")
    return None

In [18]:
legusdf["Water (g)"] = legusdf['foodNutrients'].apply(lambda items: extract_nutrient(items, "Water"))
legusdf["Calories (kcal)"] = legusdf['foodNutrients'].apply(lambda items: extract_nutrient(items, "Energy (Atwater General Factors)"))
legusdf["Nitrogen (g)"] = legusdf['foodNutrients'].apply(lambda items: extract_nutrient(items, "Nitrogen"))
legusdf["Protein (g)"] = legusdf['foodNutrients'].apply(lambda items: extract_nutrient(items, "Protein"))
legusdf["Fat (g)"] = legusdf['foodNutrients'].apply(lambda items: extract_nutrient(items, "Total lipid (fat)"))
legusdf["Ash (g)"] = legusdf['foodNutrients'].apply(lambda items: extract_nutrient(items, "Ash"))
legusdf["Carbs (g)"] = legusdf['foodNutrients'].apply(lambda items: extract_nutrient(items, "Carbohydrate, by difference"))
legusdf["Starch (g)"] = legusdf['foodNutrients'].apply(lambda items: extract_nutrient(items, "Starch"))
legusdf["Resistant starch (g)"] = legusdf['foodNutrients'].apply(lambda items: extract_nutrient(items, "Resistant starch"))
legusdf["Calcium (mg)"] = legusdf['foodNutrients'].apply(lambda items: extract_nutrient(items, "Calcium, Ca"))
legusdf["Iron (mg)"] = legusdf['foodNutrients'].apply(lambda items: extract_nutrient(items, "Iron, Fe"))
legusdf["Magnesium (mg)"] = legusdf['foodNutrients'].apply(lambda items: extract_nutrient(items, "Magnesium, Mg"))
legusdf["Phosphorus (mg)"] = legusdf['foodNutrients'].apply(lambda items: extract_nutrient(items, "Phosphorus, P"))
legusdf["Potassium (mg)"] = legusdf['foodNutrients'].apply(lambda items: extract_nutrient(items, "Potassium, K"))
legusdf["Sodium (mg)"] = legusdf['foodNutrients'].apply(lambda items: extract_nutrient(items, "Sodium, Na"))
legusdf["Zinc (mg)"] = legusdf['foodNutrients'].apply(lambda items: extract_nutrient(items, "Zinc, Zn"))
legusdf["Copper (mg)"] = legusdf['foodNutrients'].apply(lambda items: extract_nutrient(items, "Copper, Cu"))
legusdf["Manganese (mg)"] = legusdf['foodNutrients'].apply(lambda items: extract_nutrient(items, "Manganese, Mn"))


In [ ]:
legusdf = legusdf.fillna(0.0)

legusdf = legusdf.round(2)

legusdf['Category'] = legusdf["description"].str.extract(r'^(.*?),')
legusdf['Type'] = legusdf["description"].str.extract(r',\s*(.*)')


legusdf = legusdf[[
    "fdcId",
    "Category",
    "Type",
    "description",
    "Water (g)",
    "dataType",
    "publicationDate",
    "ndbNumber",
    "foodNutrients",
    "Calories (kcal)",
    "Nitrogen (g)",
    "Protein (g)",
    "Fat (g)",
    "Iron (mg)",
    "Magnesium (mg)",
    "Phosphorus (mg)",
    "Potassium (mg)",
    "Sodium (mg)",
    "Zinc (mg)",
    "Copper (mg)",
    "Manganese (mg)"
]]

legusdf = legusdf.drop(columns = ["fdcId", "description", "dataType", "ndbNumber", "foodNutrients", "publicationDate"])
legusdf


Category            object
Type                object
Water (g)          float64
Calories (kcal)    float64
Nitrogen (g)       float64
Protein (g)        float64
Fat (g)            float64
Iron (mg)          float64
Magnesium (mg)     float64
Phosphorus (mg)    float64
Potassium (mg)     float64
Sodium (mg)        float64
Zinc (mg)          float64
Copper (mg)        float64
Manganese (mg)     float64
dtype: object
